In [2]:
import json
import random
from openai import OpenAI 
import gradio as gr
import os

In [3]:
client = OpenAI(
    api_key = os.getenv("OPENAI_API_KWY")
)
MODEL = "gpt-4o-mini"

In [4]:
## dummy databases
flight_databases = {
    "AI101":{
        "from":"Hyderabad",
        "to":"Dubai",
        "status": "On Time",
        "departure": "08:30 AM"
    },
    "AI205": {
        "from": "Delhi",
        "to": "London",
        "status": "Delayed by 45 minutes",
        "departure": "10:00 PM"
    },

    "AI333": {
        "from": "Mumbai",
        "to": "Singapore",
        "status": "Boarding",
        "departure": "06:15 PM"
    }
}

passenger_details = {
    "P1001":{
        "name":"Rahul",
        "passport_valid":True
    },
    "P1002": {
        "name": "Sneha",
        "passport_valid": False
    },

    "P1003": {
        "name": "Arjun",
        "passport_valid": True
    }
}

In [9]:
## tools 
def get_weather(city):
    weather_data = {
        "Dubai": "38°C, Sunny",
        "London": "18°C, Cloudy",
        "Singapore": "31°C, Rainy",
        "Hyderabad": "33°C, Sunny",
        "Delhi": "36°C, Hot"
    }

    return weather_data.get(city, "weather unavailable")

def get_flight_status(flight_number):
    flight = flight_databases.get(flight_number)

    if flight:
        return (
            f"Flight {flight_number} is "
            f"{flight['status']}.\n"
            f"Departure: {flight['departure']}.\n"
            f"Route: {flight['from']} -> {flight['to']}"
        )

    return "Flight not found."

def check_passport(passenger_id):
    passenger = passenger_details.get(passenger_id)

    if passenger is None:
        return "passenger not found."
    if passenger["passport_valid"]:
        return (
            f"{passenger['name']}'s passport is valid."
        )

    return (
        f"{passenger['name']}'s passport has expired."
    )

def book_lounge(passenger_id):

    passenger = passenger_details.get(passenger_id)

    if passenger is None:
        return "Passenger not found."

    booking_id = random.randint(10000, 99999)

    return (
        f"Lounge booked successfully for "
        f"{passenger['name']}.\n"
        f"Booking ID : {booking_id}"
    )


In [10]:
tools = [

    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the weather of a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "Name of the city"
                    }
                },
                "required": ["city"]
            }
        }
    },

    {
        "type": "function",
        "function": {
            "name": "get_flight_status",
            "description": "Get the status of a flight.",
            "parameters": {
                "type": "object",
                "properties": {
                    "flight_number": {
                        "type": "string",
                        "description": "Flight number"
                    }
                },
                "required": ["flight_number"]
            }
        }
    },

    {
        "type": "function",
        "function": {
            "name": "check_passport",
            "description": "Check whether a passenger passport is valid.",
            "parameters": {
                "type": "object",
                "properties": {
                    "passenger_id": {
                        "type": "string",
                        "description": "Passenger ID"
                    }
                },
                "required": ["passenger_id"]
            }
        }
    },

    {
        "type": "function",
        "function": {
            "name": "book_lounge",
            "description": "Book an airport lounge for a passenger.",
            "parameters": {
                "type": "object",
                "properties": {
                    "passenger_id": {
                        "type": "string",
                        "description": "Passenger ID"
                    }
                },
                "required": ["passenger_id"]
            }
        }
    }

]

In [11]:
available_tools = {

    "get_weather": get_weather,

    "get_flight_status": get_flight_status,

    "check_passport": check_passport,

    "book_lounge": book_lounge

}

In [12]:
# Conversation History

messages = [
    {
        "role": "system",
        "content": (
            "You are an intelligent Airline Travel Assistant. "
            "Use the available tools whenever required. "
            "If the user asks for multiple tasks, call all the necessary tools "
            "before answering."
        )
    }
]


def airline_assistant(user_message):

    # ----------------------------
    # Add user message
    # ----------------------------

    messages.append(
        {
            "role": "user",
            "content": user_message
        }
    )

    # ----------------------------
    # First LLM Call
    # ----------------------------

    response = client.chat.completions.create(

        model=MODEL,

        messages=messages,

        tools=tools,

        tool_choice="auto"

    )

    assistant_message = response.choices[0].message

    # Save assistant response
    messages.append(assistant_message)

    # ---------------------------------------------------
    # If no tool is required, directly answer
    # ---------------------------------------------------

    if not assistant_message.tool_calls:

        return assistant_message.content

    # ---------------------------------------------------
    # Execute ALL requested tool calls
    # ---------------------------------------------------

    for tool_call in assistant_message.tool_calls:

        tool_name = tool_call.function.name

        arguments = json.loads(
            tool_call.function.arguments
        )

        tool_function = available_tools[tool_name]

        tool_result = tool_function(**arguments)

        # Add tool result to conversation

        messages.append(

            {
                "role": "tool",

                "tool_call_id": tool_call.id,

                "content": str(tool_result)

            }

        )

    # ---------------------------------------------------
    # Second LLM Call
    # ---------------------------------------------------

    final_response = client.chat.completions.create(

        model=MODEL,

        messages=messages

    )

    final_answer = final_response.choices[0].message.content

    messages.append(

        {
            "role": "assistant",
            "content": final_answer
        }

    )

    return final_answer

In [ ]:
import gradio as gr

demo = gr.Interface(
    fn=airline_assistant,
    inputs=gr.Textbox(
        lines=3,
        placeholder="Ask your travel-related question..."
    ),
    outputs=gr.Textbox(lines=10),
    title="AI Airline Travel Assistant",
    description="OpenAI Tool Calling Demo"
)

demo.launch()